In [1]:
import random
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder \
    .appName("BigData") \
    .config("spark.sql.catalogImplementation", "hive") \
    .enableHiveSupport() \
    .getOrCreate()

# Generate data
regions = ["Nord", "Sud", "Est", "Ouest", "Centre"]
produits = ["Laptop Pro", "Ecran 4K", "Clavier Mecanique", "Souris Sans Fil", "Casque Audio"]

data = [(i, random.choice(regions), random.choice(produits), 
         random.randint(1,5), float(random.randint(20,1200))) 
        for i in range(1000)]

df = spark.createDataFrame(data, ["transaction_id","region","produit","quantite","prix_unitaire"])
df = df.withColumn("chiffre_affaires", F.col("quantite") * F.col("prix_unitaire"))

# Clean
df = df.dropDuplicates().dropna()

# Star schema
dim_produits = df.select("produit").distinct().withColumn("produit_id", F.monotonically_increasing_id())
dim_geo = df.select("region").distinct().withColumn("region_id", F.monotonically_increasing_id())
fact_ventes = df.select("transaction_id","produit","region","quantite","prix_unitaire","chiffre_affaires")

# Save to metastore
dim_produits.write.mode("overwrite").saveAsTable("dim_produits")
dim_geo.write.mode("overwrite").saveAsTable("dim_geographie")
fact_ventes.write.mode("overwrite").saveAsTable("fact_ventes")

spark.sql("SHOW TABLES").show()
print("Done!")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/15 22:25:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/15 22:26:01 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/05/15 22:26:08 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/05/15 22:26:08 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/05/15 22:26:11 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/05/15 22:26:11 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore UNKNOWN@172.18.0.6
26/05/15 22:26:20 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manag

+---------+--------------+-----------+
|namespace|     tableName|isTemporary|
+---------+--------------+-----------+
|  default|dim_geographie|      false|
|  default|  dim_produits|      false|
|  default|   fact_ventes|      false|
+---------+--------------+-----------+

Done!


In [2]:
spark = SparkSession.builder \
    .appName("BigData") \
    .master("yarn") \
    .config("spark.sql.catalogImplementation", "hive") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .enableHiveSupport() \
    .getOrCreate()

26/05/15 22:27:55 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [1]:
spark = SparkSession.builder \
    .appName("BigData") \
    .master("yarn") \
    .config("spark.sql.catalogImplementation", "hive") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .enableHiveSupport() \
    .getOrCreate()

NameError: name 'SparkSession' is not defined

In [2]:

from pyspark.sql import SparkSession, functions as F
import random

spark = SparkSession.builder \
    .appName("BigData") \
    .master("yarn") \
    .config("spark.sql.catalogImplementation", "hive") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .enableHiveSupport() \
    .getOrCreate()


Exception in thread "main" org.apache.spark.SparkException: When running with master 'yarn' either HADOOP_CONF_DIR or YARN_CONF_DIR must be set in the environment.
	at org.apache.spark.deploy.SparkSubmitArguments.error(SparkSubmitArguments.scala:650)
	at org.apache.spark.deploy.SparkSubmitArguments.validateSubmitArguments(SparkSubmitArguments.scala:281)
	at org.apache.spark.deploy.SparkSubmitArguments.validateArguments(SparkSubmitArguments.scala:237)
	at org.apache.spark.deploy.SparkSubmitArguments.<init>(SparkSubmitArguments.scala:122)
	at org.apache.spark.deploy.SparkSubmit$$anon$2$$anon$3.<init>(SparkSubmit.scala:1108)
	at org.apache.spark.deploy.SparkSubmit$$anon$2.parseArguments(SparkSubmit.scala:1108)
	at org.apache.spark.deploy.SparkSubmit.doSubmit(SparkSubmit.scala:86)
	at org.apache.spark.deploy.SparkSubmit$$anon$2.doSubmit(SparkSubmit.scala:1125)
	at org.apache.spark.deploy.SparkSubmit$.main(SparkSubmit.scala:1134)
	at org.apache.spark.deploy.SparkSubmit.main(SparkSubmit.scal

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.